In [2]:
# Implement CNN and RNN on Titanic Dataset

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.layers import LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Titanic Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset Loaded Successfully")
print(df.head())

# 2. Select Required Features
df = df[["Survived", "Pclass", "Sex", "Age",
         "SibSp", "Parch", "Fare", "Embarked"]]

# 3. Handle Missing Values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# 4. Convert Categorical Data into Numerical Data
df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)

# 5. Separate Features and Target
X = df.drop("Survived", axis=1)
y = df["Survived"]

# 6. Split Dataset into Training and Testing Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Normalize the Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Reshape data for CNN and RNN
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 8. Build CNN Model
cnn_model = Sequential([
    Conv1D(32, kernel_size=3, activation="relu",
           padding="same", input_shape=(X_train.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nCNN Model Summary")
cnn_model.summary()

# 9. Train CNN Model
early_stop = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

cnn_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# 10. Evaluate CNN Model
cnn_prob = cnn_model.predict(X_test, verbose=0).ravel()
cnn_pred = (cnn_prob >= 0.5).astype(int)

cnn_accuracy = accuracy_score(y_test, cnn_pred)

print("\nCNN Test Accuracy:", round(cnn_accuracy * 100, 2), "%")
print("CNN Classification Report:")
print(classification_report(y_test, cnn_pred, zero_division=0))

# 11. Build RNN Model using LSTM
rnn_model = Sequential([
    LSTM(32, input_shape=(X_train.shape[1], 1)),
    Dense(16, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nRNN (LSTM) Model Summary")
rnn_model.summary()

# 12. Train RNN Model
rnn_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# 13. Evaluate RNN Model
rnn_prob = rnn_model.predict(X_test, verbose=0).ravel()
rnn_pred = (rnn_prob >= 0.5).astype(int)

rnn_accuracy = accuracy_score(y_test, rnn_pred)

print("\nRNN (LSTM) Test Accuracy:",
      round(rnn_accuracy * 100, 2), "%")
print("RNN Classification Report:")
print(classification_report(y_test, rnn_pred, zero_division=0))

# 14. Compare CNN and RNN Performance
print("\n----- MODEL COMPARISON -----")
print("CNN Accuracy:", round(cnn_accuracy * 100, 2), "%")
print("RNN (LSTM) Accuracy:", round(rnn_accuracy * 100, 2), "%")
if cnn_accuracy > rnn_accuracy:
    print("CNN performed better on this test split.")
elif rnn_accuracy > cnn_accuracy:
    print("RNN (LSTM) performed better on this test split.")
else:
   print("Both models achieved the same accuracy.")

Dataset Loaded Successfully
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 10, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 5, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,313 (20.75 KB)

 Trainable params: 5,313 (20.75 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,897 (19.13 KB)

 Trainable params: 4,897 (19.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.5923 - loss: 0.6894 - val_accuracy: 0.6084 - val_loss: 0.6840
Epoch 2/15
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6239 - loss: 0.6754 - val_accuracy: 0.6014 - val_loss: 0.6734
Epoch 3/15
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6344 - loss: 0.6593 - val_accuracy: 0.6014 - val_loss: 0.6653

RNN (LSTM) Test Accuracy: 61.45 %
RNN Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.91      0.74       110
           1       0.50      0.14      0.22        69

    accuracy                           0.61       179
   macro avg       0.56      0.53      0.48       179
weighted avg       0.58      0.61      0.54       179


----- MODEL COMPARISON -----
CNN Accuracy: 80.45 %
RNN (LSTM) Accuracy: 61.45 %
CNN performed better on this test split.
